# Relative and absolute measures of effect in clinical research: interpretation of risk metrics and their ratios with computational implementation in r and python

**Authors:** Renato Carneiro de Freitas Chaves, Tiago Mendonça dos Santos, Thiago Domingos Corrêa  

## 1) Purpose

This notebook provides a step-by-step structure to calculate effect measures for a **binary outcome** comparing:

- **Control group** (`group = 0`)
- **Intervention group** (`group = 1`)

Effect measures:

- **Relative Risk (RR)**
- **Relative Risk Reduction (RRR)**
- **Absolute Risk Reduction (ARR)**
- **Number Needed to Treat (NNT)**
- **Number Needed to Harm (NNH)**

### Outcome definition in this dataset

In the dataset model (`data.xlsx`), the binary endpoint is:

- `outcome`: `0 = Alive`, `1 = Dead`

We treat **death** (`outcome = 1`) as the **event of interest**.

> **Interpretation rule**  
> - If the intervention **reduces** event risk, report **NNT**.  
> - If the intervention **increases** event risk, report **NNH**.


## 2) Required Libraries

This notebook uses standard scientific Python libraries.


In [20]:
import math
import numpy as np
import pandas as pd
from scipy.stats import norm

# Display options (optional)
pd.set_option("display.max_rows", 200)
pd.set_option("display.max_columns", 50)


## 3) Data Import (Excel)

This guide uses the Excel file **`data.xlsx`** with a sheet named **`Data`**.

- If `data.xlsx` is in the same folder as this notebook, you can keep the path as `"data.xlsx"`.
- Otherwise, replace with the full path to your file.


In [21]:
# Update the path if needed
excel_path = "data.xlsx"
sheet_name = "Data"

data = pd.read_excel(excel_path, sheet_name=sheet_name)

# Quick inspection
data.head(), data.shape

(   patient  group  center  age  platelet  hemoglobin  creatinine  \
 0        1      1       1   19       229        15.3         0.8   
 1        2      1       1   24       221        14.8         0.8   
 2        3      1       1   25       219        14.4         0.9   
 3        4      1       1   26       217        14.3         0.9   
 4        5      1       1   27       214        13.8         0.9   
 
    hypertension  diabetes  heart_failure  smoking  chronic_kidney_disease  \
 0             0         0              0        0                       0   
 1             0         0              0        0                       0   
 2             0         0              0        0                       0   
 3             0         0              0        0                       0   
 4             1         0              1        0                       1   
 
    cancer  stroke  saps_3  vasopressor  mechanical_ventilation   \
 0       0       0       5            0       

In [22]:
# Column names and basic info

data.info()

list(data.columns)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50 entries, 0 to 49
Data columns (total 19 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   patient                  50 non-null     int64  
 1   group                    50 non-null     int64  
 2   center                   50 non-null     int64  
 3   age                      50 non-null     int64  
 4   platelet                 50 non-null     int64  
 5   hemoglobin               50 non-null     float64
 6   creatinine               50 non-null     float64
 7   hypertension             50 non-null     int64  
 8   diabetes                 50 non-null     int64  
 9   heart_failure            50 non-null     int64  
 10  smoking                  50 non-null     int64  
 11  chronic_kidney_disease   50 non-null     int64  
 12  cancer                   50 non-null     int64  
 13  stroke                   50 non-null     int64  
 14  saps_3                   50 

['patient',
 'group',
 'center',
 'age',
 'platelet',
 'hemoglobin',
 'creatinine',
 'hypertension',
 'diabetes',
 'heart_failure',
 'smoking',
 'chronic_kidney_disease',
 'cancer',
 'stroke',
 'saps_3',
 'vasopressor',
 'mechanical_ventilation ',
 'fluid_balance',
 'outcome']

## 4) Confirm Required Variables

For the requested calculations, we need:

- `group` (0 = Control, 1 = Intervention)
- `outcome` (0 = Alive, 1 = Dead)


In [23]:
required_cols = {"group", "outcome"}
missing = required_cols - set(data.columns)

if missing:
    raise ValueError(f"Missing required column(s): {sorted(missing)}")

# Frequency checks
data["group"].value_counts(dropna=False), data["outcome"].value_counts(dropna=False)

(1    25
 0    25
 Name: group, dtype: int64,
 1    30
 0    20
 Name: outcome, dtype: int64)

## 5) Create the Analysis Dataset

We will:

- keep only rows with non-missing `group` **and** `outcome`
- ensure they are coded as expected (0/1)
- define the event as `outcome == 1` (death)


In [24]:
analysis_data = (
    data.loc[:, ["group", "outcome"]]
        .dropna(subset=["group", "outcome"])
        .assign(
            group=lambda df: df["group"].astype(int),
            event=lambda df: df["outcome"].astype(int)  # event = 1 means "death"
        )
)

analysis_data.head(), analysis_data.shape

(   group  outcome  event
 0      1        0      0
 1      1        0      0
 2      1        0      0
 3      1        0      0
 4      1        0      0,
 (50, 3))

## 6) Construct the 2×2 Table

Rows:
- Intervention
- Control

Columns:
- Event (Dead)
- Non-event (Alive)


In [25]:
analysis_data = analysis_data.assign(
    group_label=pd.Categorical(
        analysis_data["group"],
        categories=[1, 0],
        ordered=True
    ).rename_categories(["Intervention", "Control"]),
    event_label=pd.Categorical(
        analysis_data["event"],
        categories=[1, 0],
        ordered=True
    ).rename_categories(["Event", "Non-event"])
)

tab = pd.crosstab(analysis_data["group_label"], analysis_data["event_label"])
tab

event_label,Event,Non-event
group_label,,
Intervention,11,14
Control,19,6


## 7) Compute Risks and Effect Measures

### 7.1 Extract cell counts (a, b, c, d)

We define:

- `a` = Events in **Intervention**
- `b` = Non-events in **Intervention**
- `c` = Events in **Control**
- `d` = Non-events in **Control**


In [26]:
a = int(tab.loc["Intervention", "Event"])
b = int(tab.loc["Intervention", "Non-event"])
c = int(tab.loc["Control", "Event"])
d = int(tab.loc["Control", "Non-event"])

a, b, c, d

(11, 14, 19, 6)

### 7.2 Compute risks (event rates)

\[
Risk_{Intervention} = \frac{a}{a+b}, \quad Risk_{Control} = \frac{c}{c+d}
\]


In [27]:
risk_intervention = a / (a + b)
risk_control = c / (c + d)

risk_intervention, risk_control

(0.44, 0.76)

### 7.3 Relative Risk (RR)

\[
RR = \frac{Risk_{Intervention}}{Risk_{Control}}
\]


In [28]:
RR = risk_intervention / risk_control
RR

0.5789473684210527

### 7.4 Relative Risk Reduction (RRR)

When the event is undesirable (death), a common definition is:

\[
RRR = 1 - RR
\]


In [29]:
RRR = 1 - RR
RRR

0.42105263157894735

### 7.5 Absolute Risk Reduction (ARR)

\[
ARR = Risk_{Control} - Risk_{Intervention}
\]


In [30]:
ARR = risk_control - risk_intervention
ARR

0.32

### 7.6 NNT and NNH

- If `ARR > 0`, the intervention reduces risk → **NNT = 1 / ARR**
- If `ARR < 0`, the intervention increases risk → **NNH = 1 / |ARR|**
- If `ARR = 0`, neither NNT nor NNH is defined (infinite)


In [31]:
if ARR > 0:
    NNT = 1 / ARR
    NNH = np.nan
elif ARR < 0:
    NNT = np.nan
    NNH = 1 / abs(ARR)
else:
    NNT = math.inf
    NNH = math.inf

NNT, NNH

(3.125, nan)

## 8) Confidence Intervals for Risk Difference and Relative Risk

For reporting, the **absolute risk reduction (ARR)** and **relative risk (RR)** are typically presented with **95% confidence intervals (CIs)**.

- **ARR** is reported as:  
  \[
  ARR = Risk_{Control} - Risk_{Intervention}
  \]
- **RR** is reported as:  
  \[
  RR = \frac{Risk_{Intervention}}{Risk_{Control}}
  \]


### 8.1 Absolute Risk Reduction (ARR) with 95% CI

A common large-sample (Wald-type) CI for the difference in proportions is:

\[
ARR \pm z_{1-\alpha/2} \cdot SE,
\quad
SE = \sqrt{\frac{p_C(1-p_C)}{n_C} + \frac{p_I(1-p_I)}{n_I}}
\]

This implementation uses **no continuity correction**.


In [32]:
n_i = a + b
n_c = c + d

p_i = risk_intervention
p_c = risk_control

ARR = p_c - p_i

alpha = 0.05
z = norm.ppf(1 - alpha / 2)

SE_arr = math.sqrt((p_c * (1 - p_c) / n_c) + (p_i * (1 - p_i) / n_i))
ARR_CI_low = ARR - z * SE_arr
ARR_CI_high = ARR + z * SE_arr

ARR, ARR_CI_low, ARR_CI_high

(0.32, 0.0633120538619984, 0.5766879461380017)

### 8.2 Relative Risk (RR) with 95% CI

A log (Wald) CI is computed for the RR. To improve stability when any cell count is zero, a continuity correction of **0.5** is applied to all four cells if **any** cell is zero.

\[
SE_{\log(RR)} = \sqrt{\left(\frac{1}{a}-\frac{1}{a+b}\right) + \left(\frac{1}{c}-\frac{1}{c+d}\right)}
\]

\[
CI_{RR} = \exp\left(\log(RR) \pm z_{1-\alpha/2} \cdot SE_{\log(RR)}\right)
\]


In [33]:
cc = 0.5 if any(x == 0 for x in (a, b, c, d)) else 0.0

a_cc = a + cc
b_cc = b + cc
c_cc = c + cc
d_cc = d + cc

risk_intervention_cc = a_cc / (a_cc + b_cc)
risk_control_cc = c_cc / (c_cc + d_cc)

RR_cc = risk_intervention_cc / risk_control_cc

SE_logRR = math.sqrt(
    (1 / a_cc) - (1 / (a_cc + b_cc)) +
    (1 / c_cc) - (1 / (c_cc + d_cc))
)

RR_CI_low = math.exp(math.log(RR_cc) - z * SE_logRR)
RR_CI_high = math.exp(math.log(RR_cc) + z * SE_logRR)

pd.DataFrame({
    "rr": [RR_cc],
    "rr_ci_low": [RR_CI_low],
    "rr_ci_high": [RR_CI_high],
    "continuity_correction_applied": [cc]
})

,rr,rr_ci_low,rr_ci_high,continuity_correction_applied
0,0.578947,0.353244,0.948864,0.0


> **Quality control tip:** Confirm that the “Event” column corresponds to the adverse outcome you intend to analyze (here: `outcome = 1`, death).

## 9) Final Summary Table

In [34]:
# A compact, publication table with CI columns where applicable
final_table = pd.DataFrame([
    {"measure": "Risk (Control)", "value": risk_control, "lower_ci": np.nan, "upper_ci": np.nan},
    {"measure": "Risk (Intervention)", "value": risk_intervention, "lower_ci": np.nan, "upper_ci": np.nan},
    {"measure": "Absolute Risk Reduction (ARR = Rc - Ri)", "value": ARR, "lower_ci": ARR_CI_low, "upper_ci": ARR_CI_high},
    {"measure": "Relative Risk (RR)", "value": RR_cc, "lower_ci": RR_CI_low, "upper_ci": RR_CI_high},
    {"measure": "Relative Risk Reduction (RRR = 1 - RR)", "value": RRR, "lower_ci": np.nan, "upper_ci": np.nan},
    {"measure": "Number Needed to Treat (NNT) [if ARR > 0]", "value": NNT, "lower_ci": np.nan, "upper_ci": np.nan},
    {"measure": "Number Needed to Harm (NNH) [if ARR < 0]", "value": NNH, "lower_ci": np.nan, "upper_ci": np.nan},
])

final_table.round(3)

,measure,value,lower_ci,upper_ci
0,Risk (Control),0.760,NaN,NaN
1,Risk (Intervention),0.440,NaN,NaN
2,Absolute Risk Reduction (ARR = Rc - Ri),0.320,0.063,0.577
3,Relative Risk (RR),0.579,0.353,0.949
4,Relative Risk Reduction (RRR = 1 - RR),0.421,NaN,NaN
5,Number Needed to Treat (NNT) [if ARR > 0],3.125,NaN,NaN
6,Number Needed to Harm (NNH) [if ARR < 0],NaN,NaN,NaN


---

## End of notebook